# Rhythmic Motion Diffusion - Interactive Demo

**MDM + Audio Cross-Attention + Beat-Aware Bias**  
Model: `wav2clip_beataware` · ag=1.5, tg=2.5 · 20 FPS · HumanML3D 263-dim

| Section | Description |
|---------|-------------|
| **§1 Setup** | Load model, diffusion, normalization stats |
| **§2 Constants & Helpers** | Genres, kinematic chains, render utilities |
| **§3 Text Sweep** | Fixed audio (Jazz Swing 110 BPM) · 3 text prompts |
| **§4 Audio Sweep** | Fixed prompt · 3 music genres (dropdown) |
| **§5 Baseline Comparisons** | Text-only (MDM vs Ours) · Audio-only (EDGE vs Ours) |
| **§6 Free-form** | Custom prompt · audio · model · guidance |

In [11]:
import os, sys, json, subprocess
import numpy as np
import torch
from copy import deepcopy
from types import SimpleNamespace
from scipy.ndimage import gaussian_filter1d
from IPython.display import display, Video, HTML
import ipywidgets as widgets
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.lines import Line2D
import librosa
import imageio

sys.path.insert(0, '.')
os.environ.setdefault('PYOPENGL_PLATFORM', 'egl')
os.environ.setdefault('NUMBA_CACHE_DIR', '/tmp/numba_cache')

'/tmp/numba_cache'

## §1 · Setup - Load Model & Diffusion

Run once before any experiment. Detects `./final_weights/` (Google Drive) and falls back to `./save/` automatically.

In [12]:
# ── Google Drive weights ───────────────────────────────────────────────────────
# Download final_weights/ from:
#   https://drive.google.com/drive/folders/1CgdWvdJcV2QTfV_lmHDF2QXhWI4dtEK-
# Place it at the repo root.  If not found, falls back to local dev paths.

WEIGHTS_ROOT = './final_weights'
USE_DRIVE_WEIGHTS = os.path.isdir(WEIGHTS_ROOT)

def _resolve(drive_path, fallback_path):
    """Return drive_path if it exists, else fallback_path."""
    full = os.path.join(WEIGHTS_ROOT, drive_path) if USE_DRIVE_WEIGHTS else None
    if full and os.path.exists(full):
        return full
    return fallback_path

# ── paths ──────────────────────────────────────────────────────────────────────
MODEL_PATH = _resolve(
    'stage2/audio_stage2_wav2clip_beataware/model_final.pt',
    './save/audio_stage2_wav2clip_beataware/model_final.pt',
)
HUMANML_DIR = './dataset/HumanML3D'
AUDIO_DIR   = './dataset/aist/audio_test_10'
OUTPUT_DIR  = './demo_outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using {"Google Drive" if USE_DRIVE_WEIGHTS else "local dev"} weights')
print(f'Device: {DEVICE}')

# ── normalization stats ────────────────────────────────────────────────────────
MEAN = np.load(os.path.join(HUMANML_DIR, 'Mean.npy'))
STD  = np.load(os.path.join(HUMANML_DIR, 'Std.npy'))

# ── load model ─────────────────────────────────────────────────────────────────
from sample.generate_audio import load_model, AudioCFGSampleModel
from utils.model_util import create_gaussian_diffusion

model, ckpt_args = load_model(MODEL_PATH, DEVICE)

diffusion = create_gaussian_diffusion(SimpleNamespace(
    diffusion_steps=1000, noise_schedule='cosine', sigma_small=True,
    lambda_vel=0.0, lambda_rcxyz=0.0, lambda_fc=0.0, lambda_target_loc=0.0,
))

# ── base MDM (text-only, no audio conditioning) ───────────────────────────────
BASE_MDM_PATH = _resolve(
    'pretrained/humanml_trans_enc_512/model000200000.pt',
    '/Data/yash.bhardwaj/pretrained/humanml_trans_enc_512/model000200000.pt',
)

# ── pre-generated EDGE outputs (SMPL 24-joint pkl files) ──────────────────────
EDGE_MOTION_DIR = _resolve(
    'pretrained/edge/edge_bas_motions',
    '/Data/yash.bhardwaj/eval/edge_bas/motions',
)

print('Model and diffusion ready.')

Using Google Drive weights
Device: cuda
AUDIO CONDITIONING enabled
TRANS_ENC init
EMBED TEXT
Loading CLIP...
Model loaded from ./save/audio_stage2_wav2clip_beataware/model_final.pt
Model and diffusion ready.


## §2 · Constants & Helpers

Genre table, kinematic chains, rendering utilities. Run once — no output.

In [13]:
# ── constants ──────────────────────────────────────────────────────────────────
FPS = 20

GENRES = {
    'mBR0': ('Breakdance',     161),
    'mJS3': ('Jazz Swing',     110),
    'mKR2': ('Krump',          122),
    'mLH4': ('LA Hiphop',      105),
    'mLO2': ('Lock',           118),
    'mMH3': ('Middle Hiphop',  100),
    'mPO1': ('Pop',            116),
    'mWA0': ('Waacking',        81),
}

GENRE_OPTIONS = {f"{v[0]} ({v[1]} BPM)": k for k, v in GENRES.items()}

KINEMATIC_CHAINS = [
    [0, 2, 5, 8, 11], [0, 1, 4, 7, 10],
    [0, 3, 6, 9, 12, 15], [9, 14, 17, 19, 21], [9, 13, 16, 18, 20],
]

CHAIN_COLORS = ['#4A90E2', '#7B68EE', '#F5A623', '#FF6B6B', '#50C878']
LINEWIDTHS   = [3.0, 3.0, 3.5, 2.5, 2.5]

BG    = '#0D0D0D'
BG2   = '#0C0C1A'
AMBER = '#FFB300'
MINT  = '#80CBC4'
FLOOR = '#B0B0B0'

# SMPL 24-joint kinematic chains (for EDGE outputs)
SMPL_CHAINS = [
    [0,1,4,7,10], [0,2,5,8,11], [0,3,6,9,12,15],
    [9,14,17,19,21,23], [9,13,16,18,20,22],
]
SMPL_COLORS = ['#E27A4A', '#EE8B68', '#F5C623', '#FF6B6B', '#50C878']
SMPL_LW     = [3.0, 3.0, 3.5, 2.5, 2.5]
EDGE_FPS    = 30

from eval.beat_align_score import (
    load_humanml_motion, get_music_beats, get_motion_beats,
    ba_score, BA_SIGMA_TIME_SEC, SMOOTH_TIME_SEC,
    recover_joints_from_humanml,
)

In [14]:
# ══════════════════════════════════════════════════════════════════════════════
#  Core helpers: generate, recover joints, render grid
# ══════════════════════════════════════════════════════════════════════════════

import pickle

# ── Base MDM loader (no audio conditioning) ──────────────────────────────────

_base_mdm_cache = {}

def load_base_mdm(model_path, device):
    """Load the original MDM (text-only, no audio conditioning)."""
    if model_path in _base_mdm_cache:
        return _base_mdm_cache[model_path]
    ckpt = torch.load(model_path, map_location='cpu')
    args_path = os.path.join(os.path.dirname(model_path), 'args.json')
    with open(args_path, 'r') as f:
        base_args = json.load(f)
    from model.mdm import MDM
    m = MDM(
        modeltype='', njoints=263, nfeats=1, num_actions=1, translation=True,
        pose_rep='rot6d', glob=True, glob_rot=[3.141592653589793, 0, 0],
        latent_dim=base_args.get('latent_dim', 512),
        ff_size=base_args.get('ff_size', 1024),
        num_layers=base_args.get('layers', 8),
        num_heads=base_args.get('num_heads', 4),
        dropout=base_args.get('dropout', 0.1),
        activation=base_args.get('activation', 'gelu'),
        data_rep='hml_vec', dataset='humanml', clip_dim=512, arch='trans_enc',
        clip_version=base_args.get('clip_version', 'ViT-B/32'),
        cond_mode='text', cond_mask_prob=base_args.get('cond_mask_prob', 0.1),
        audio_conditioning=False,
    )
    state = ckpt['model'] if 'model' in ckpt else ckpt
    m.load_state_dict(state, strict=False)
    m.to(device)
    m.eval()
    _base_mdm_cache[model_path] = m
    print(f'Base MDM loaded from {model_path}')
    return m


def generate_motion_base_mdm(text_prompt, tg=2.5, seed=42, n_frames=196):
    """Generate motion using the original MDM (text-only, no audio)."""
    from model.cfg_sampler import ClassifierFreeSampleModel
    torch.manual_seed(seed)
    np.random.seed(seed)
    base_model = load_base_mdm(BASE_MDM_PATH, DEVICE)
    cfg = ClassifierFreeSampleModel(base_model)
    model_kwargs = {
        'y': {
            'text': [text_prompt],
            'mask': torch.ones(1, 1, 1, n_frames, dtype=torch.bool).to(DEVICE),
            'lengths': torch.tensor([n_frames]).to(DEVICE),
            'scale': torch.tensor([tg]).to(DEVICE),
        }
    }
    with torch.no_grad():
        sample = diffusion.p_sample_loop(
            cfg, (1, 263, 1, n_frames),
            clip_denoised=False, model_kwargs=model_kwargs,
            skip_timesteps=0, init_image=None, progress=True,
        )
    motion = sample.squeeze(2).permute(0, 2, 1).cpu().numpy()[0]
    motion = motion * STD + MEAN
    motion = gaussian_filter1d(motion, sigma=1.0, axis=0)
    return motion


# ── EDGE joint loading + prep ────────────────────────────────────────────────

def load_edge_joints(song_id):
    """Load pre-generated EDGE pkl → (T, 24, 3) in y-up convention."""
    pkl_path = os.path.join(EDGE_MOTION_DIR, f'test_{song_id}.pkl')
    with open(pkl_path, 'rb') as f:
        data = pickle.load(f)
    fp = data['full_pose'].astype(np.float32)  # (T, 24, 3) z-up
    joints = np.zeros_like(fp)
    joints[:,:,0] = fp[:,:,0]
    joints[:,:,1] = fp[:,:,2]   # z → y (height)
    joints[:,:,2] = fp[:,:,1]   # y → z (depth)
    return joints


def _prep_cell_edge(joints_raw, aud_path, fps=30):
    """Prepare rendering data from EDGE (24-joint) output at given fps."""
    T = joints_raw.shape[0]
    j = joints_raw.copy() * SCALE
    j[:, :, 1] -= j[:, :, 1].min()
    trajec = j[:, 0, [0, 2]].copy()
    jd = j.copy()
    jd[:, :, 0] -= jd[:, 0:1, 0]
    jd[:, :, 2] -= jd[:, 0:1, 2]
    MINS = j.min(0).min(0)
    MAXS = j.max(0).max(0)
    dur = T / fps

    vel = np.mean(np.sqrt(np.sum((j[1:] - j[:-1])**2, axis=2)), axis=1)
    vel_s = gaussian_filter1d(vel, SMOOTH_TIME_SEC * fps)
    vel_norm = vel_s / (vel_s.max() + 1e-8) * 0.4

    has_audio = aud_path is not None and os.path.isfile(aud_path)
    if has_audio:
        y_full, sr = librosa.load(aud_path, sr=None)
        y_audio = y_full[:int(dur * sr)]
        music_beats = get_music_beats(aud_path, fps, max_frames=T)
    else:
        sr = 22050; y_audio = np.zeros(int(dur * sr))
        music_beats = np.array([], dtype=float)

    motion_beats = get_motion_beats(joints_raw, fps=fps)
    bas = ba_score(music_beats, motion_beats, sigma=BA_SIGMA_TIME_SEC * fps) if has_audio else None

    return dict(
        joints_disp=jd, trajec=trajec, T=T, duration=dur, fps=fps,
        MINS=MINS, MAXS=MAXS, y=y_audio, sr=sr,
        music_beats=music_beats, motion_beats=motion_beats,
        bas=bas, vel_norm=vel_norm, has_audio=has_audio,
        is_edge=True, chains=SMPL_CHAINS, colors=SMPL_COLORS, linewidths=SMPL_LW,
    )


def extract_audio(wav_path, n_frames=196):
    """Extract 519-dim wav2clip+librosa features and beat frames."""
    from model.audio_features_wav2clip import extract_wav2clip_plus_librosa
    feat = extract_wav2clip_plus_librosa(wav_path, target_fps=FPS, duration=None, device=DEVICE)
    feat = feat[:n_frames]
    beat_idx = 513
    beat_frames = list(np.where(feat[:, beat_idx] > 0.5)[0])
    return torch.from_numpy(feat).float(), beat_frames


def generate_motion(text_prompt, audio_path=None, tg=2.5, ag=1.5, seed=42, n_frames=196):
    """Generate a single (T, 263) denormalized motion array."""
    torch.manual_seed(seed)
    np.random.seed(seed)

    cfg_model = AudioCFGSampleModel(model, text_scale=tg, audio_scale=ag)

    audio_features = None
    beat_frames = []

    if audio_path and os.path.isfile(audio_path):
        af, beat_frames = extract_audio(audio_path, n_frames)
        n_frames = min(af.shape[0], n_frames)
        af = af[:n_frames]
        audio_features = af.unsqueeze(0).to(DEVICE)

    model_kwargs = {
        'y': {
            'text': [text_prompt],
            'mask': torch.ones(1, 1, 1, n_frames, dtype=torch.bool).to(DEVICE),
            'lengths': torch.tensor([n_frames]).to(DEVICE),
            'scale': torch.tensor([tg]).to(DEVICE),
        }
    }
    if audio_features is not None:
        model_kwargs['y']['audio_features'] = audio_features
        model_kwargs['y']['beat_frames'] = beat_frames

    with torch.no_grad():
        sample = diffusion.p_sample_loop(
            cfg_model, (1, 263, 1, n_frames),
            clip_denoised=False, model_kwargs=model_kwargs,
            skip_timesteps=0, init_image=None, progress=True,
            dump_steps=None, noise=None, const_noise=False,
        )

    motion = sample.squeeze(2).permute(0, 2, 1).cpu().numpy()[0]  # (T, 263)
    motion = motion * STD + MEAN
    motion = gaussian_filter1d(motion, sigma=1.0, axis=0)
    return motion


def motion_to_joints(motion_263):
    """(T, 263) denormalized -> (T, 22, 3) joint positions."""
    return recover_joints_from_humanml(motion_263)


SCALE  = 1.3
RADIUS = 3

def _prep_cell(joints_raw, aud_path=None):
    """Prepare rendering data dict from joint positions."""
    T = joints_raw.shape[0]
    j = joints_raw.copy() * SCALE
    j[:, :, 1] -= j[:, :, 1].min()
    trajec = j[:, 0, [0, 2]].copy()
    jd = j.copy()
    jd[:, :, 0] -= jd[:, 0:1, 0]
    jd[:, :, 2] -= jd[:, 0:1, 2]
    MINS = j.min(0).min(0)
    MAXS = j.max(0).max(0)
    dur = T / FPS

    vel = np.mean(np.sqrt(np.sum((j[1:] - j[:-1])**2, axis=2)), axis=1)
    vel_s = gaussian_filter1d(vel, SMOOTH_TIME_SEC * FPS)
    vel_norm = vel_s / (vel_s.max() + 1e-8) * 0.4

    has_audio = aud_path is not None and os.path.isfile(aud_path)
    if has_audio:
        y_full, sr = librosa.load(aud_path, sr=None)
        y_audio = y_full[:int(dur * sr)]
        music_beats = get_music_beats(aud_path, FPS, max_frames=T)
    else:
        sr = 22050; y_audio = np.zeros(int(dur * sr))
        music_beats = np.array([], dtype=float)

    motion_beats = get_motion_beats(joints_raw, fps=FPS)
    bas = ba_score(music_beats, motion_beats, sigma=BA_SIGMA_TIME_SEC * FPS) if has_audio else None

    return dict(
        joints_disp=jd, trajec=trajec, T=T, duration=dur,
        MINS=MINS, MAXS=MAXS, y=y_audio, sr=sr,
        music_beats=music_beats, motion_beats=motion_beats,
        bas=bas, vel_norm=vel_norm, has_audio=has_audio,
    )


def _draw_cell(fig, col_gs, cd, fi, t_now, mbeat_color='#F06292'):
    inner = gridspec.GridSpecFromSubplotSpec(2, 1, subplot_spec=col_gs,
                                             height_ratios=[3.2, 1], hspace=0.04)
    ax3d  = fig.add_subplot(inner[0], projection='3d')
    ax_wav = fig.add_subplot(inner[1])

    ax3d.set_facecolor(BG)
    jf = cd['joints_disp'][fi]
    cx, cz = cd['trajec'][fi]
    MINS, MAXS = cd['MINS'], cd['MAXS']
    nz = lambda z: -z

    gx = np.linspace(MINS[0]-cx, MAXS[0]-cx, 9)
    gz = np.linspace(nz(MAXS[2]-cz), nz(MINS[2]-cz), 9)
    for gxi in gx:
        ax3d.plot([gxi,gxi],[0,0],[gz[0],gz[-1]], color=FLOOR, lw=0.7, alpha=0.55)
    for gzi in gz:
        ax3d.plot([gx[0],gx[-1]],[0,0],[gzi,gzi], color=FLOOR, lw=0.7, alpha=0.55)

    ts = max(0, fi-20)
    tp = cd['trajec'][ts:fi+1]
    if len(tp)>1:
        tx=tp[:,0]-cx; tz=nz(tp[:,1]-cz)
        alphas=np.linspace(0.1,0.7,len(tp))
        for k in range(len(tp)-1):
            ax3d.plot([tx[k],tx[k+1]],[0.02,0.02],[tz[k],tz[k+1]],
                      color='#5599FF', lw=2.0, alpha=float(alphas[k+1]))

    chains = cd.get('chains', KINEMATIC_CHAINS)
    colors = cd.get('colors', CHAIN_COLORS)
    lws    = cd.get('linewidths', LINEWIDTHS)
    for chain, color, lw in zip(chains, colors, lws):
        ax3d.plot3D(jf[chain,0], jf[chain,1], nz(jf[chain,2]),
                    color=color, linewidth=lw, marker='o', markersize=2.5, markerfacecolor='white')

    r = RADIUS
    ax3d.set_xlim3d([-r/2,r/2]); ax3d.set_ylim3d([0,r]); ax3d.set_zlim3d([-r/3,r*2/3])
    ax3d.view_init(elev=120, azim=-90); ax3d.dist = 7.5
    ax3d.set_axis_off(); ax3d.grid(False)
    ax3d.set_xticks([]); ax3d.set_yticks([]); ax3d.set_zticks([])
    for pane in (ax3d.xaxis.pane, ax3d.yaxis.pane, ax3d.zaxis.pane):
        pane.fill=False; pane.set_edgecolor(BG)

    bas_txt = f"BAS {cd['bas']:.3f}" if cd['bas'] is not None else 'text-only'
    ax3d.text2D(0.97, 0.97, bas_txt, transform=ax3d.transAxes,
                color='#FFD700', fontsize=8, ha='right', va='top', fontfamily='monospace',
                bbox=dict(boxstyle='round,pad=0.25', fc=BG, ec='#665500', alpha=0.88))

    ax_wav.set_facecolor(BG2)
    dur = cd['duration']
    times = np.linspace(0, dur, len(cd['y']))
    wav_max = max(np.abs(cd['y']).max()*1.1, 0.01)
    vel_t = np.arange(len(cd['vel_norm'])) / FPS

    ax_wav.fill_between(times, cd['y'], alpha=0.15, color='#7986CB')
    ax_wav.fill_between(times, -cd['y'], alpha=0.15, color='#7986CB')
    ax_wav.plot(times, cd['y'], color='#9FA8DA', lw=0.4, alpha=0.4)
    ax_wav.axhline(0, color='#333355', lw=0.6, alpha=0.5)

    if cd['has_audio']:
        for bt in cd['music_beats']/FPS:
            ax_wav.axvline(bt, color=AMBER, lw=1.5, alpha=0.92, zorder=5)
    for bt in cd['motion_beats']/FPS:
        ax_wav.axvline(bt, color=mbeat_color, lw=1.5, alpha=0.92, zorder=6)

    ax_wav.plot(vel_t, cd['vel_norm'], color=MINT, lw=1.3, alpha=0.88, zorder=4)
    ax_wav.axvline(t_now, color='#FFFFFF', lw=1.8, alpha=1.0, zorder=8)
    ax_wav.set_xlim(0, dur); ax_wav.set_ylim(-wav_max, wav_max)
    ax_wav.set_xticks([]); ax_wav.set_yticks([])
    for sp in ax_wav.spines.values(): sp.set_edgecolor('#1E1E33')

    return ax3d


def render_grid_video(cells, col_labels, mbeat_colors, title, out_path, aud_path=None, clip_sec=8.0, render_fps=None):
    """Render a 1xN grid video from a list of (cell_data, label) and save to mp4.
    Handles mixed FPS cells by mapping output time → each cell's native frame index."""
    N = len(cells)
    CELL_W, CELL_H = 5.2, 4.8
    FIG_W, FIG_H = N * CELL_W, CELL_H + 0.5
    DPI = 100
    if render_fps is None:
        render_fps = FPS
    min_dur = min(cd['duration'] for cd in cells)
    clip_dur = min(min_dur, clip_sec)
    T_render = int(clip_dur * render_fps)

    frames_rgb = []
    for out_fi in range(T_render):
        t_now = out_fi / render_fps
        fig = plt.figure(figsize=(FIG_W, FIG_H), dpi=DPI, facecolor=BG)
        outer = gridspec.GridSpec(1, N, figure=fig, hspace=0.0, wspace=0.04,
                                  left=0.01, right=0.99, top=0.91, bottom=0.02)
        ax_refs = []
        for col in range(N):
            cell_fps = cells[col].get('fps', FPS)
            fi = min(int(t_now * cell_fps), cells[col]['T'] - 1)
            ax3d = _draw_cell(fig, outer[0,col], cells[col], fi, t_now, mbeat_colors[col])
            ax_refs.append(ax3d)

        fig.canvas.draw()
        for ax3d, lbl in zip(ax_refs, col_labels):
            pos = ax3d.get_position()
            fig.text((pos.x0+pos.x1)/2, pos.y1+0.005, lbl,
                     color='#DDDDDD', fontsize=8.5, ha='center', va='bottom',
                     fontfamily='monospace', fontweight='bold')

        fig.suptitle(f'{title}  |  t={t_now:.2f}s',
                     color='#DDDDDD', fontsize=8.5, y=0.98, fontfamily='monospace')

        legend_elems = [
            Line2D([0],[0], color=AMBER, lw=1.8, label='Music beats'),
            Line2D([0],[0], color=mbeat_colors[0], lw=1.8, label='Motion beats'),
            Line2D([0],[0], color=MINT, lw=1.4, label='Velocity'),
            Line2D([0],[0], color='#FFFFFF', lw=1.6, label='Playhead'),
        ]
        fig.legend(handles=legend_elems, loc='lower right', fontsize=7.2,
                   facecolor=BG2, edgecolor='#2A2A45', labelcolor='#C5CAE9',
                   ncol=4, framealpha=0.92, bbox_to_anchor=(0.99, 0.0))

        fig.canvas.draw()
        w_px, h_px = fig.canvas.get_width_height()
        buf = np.frombuffer(fig.canvas.buffer_rgba(), dtype=np.uint8)
        buf = buf.reshape(h_px, w_px, 4)[:,:,:3]
        buf = buf[:buf.shape[0]//2*2, :buf.shape[1]//2*2]
        frames_rgb.append(buf.copy())
        plt.close(fig)
        if out_fi % 40 == 0:
            print(f'  frame {out_fi+1}/{T_render}')

    silent = out_path.replace('.mp4', '_silent.mp4')
    writer = imageio.get_writer(silent, fps=render_fps, codec='libx264', quality=8, macro_block_size=1)
    for f in frames_rgb:
        writer.append_data(f)
    writer.close()

    if aud_path and os.path.isfile(aud_path):
        r = subprocess.run(['ffmpeg','-y','-i',silent,'-i',aud_path,
                            '-c:v','copy','-c:a','aac','-b:a','192k','-shortest',out_path],
                           capture_output=True)
        if r.returncode == 0:
            os.remove(silent)
        else:
            os.rename(silent, out_path)
    else:
        os.rename(silent, out_path)

    print(f'  Saved → {out_path}')
    return out_path

## §3 · Text Sweep

Fixed audio: **Jazz Swing** (mJS3, 110 BPM) · Prompts: walk / jump / kick · ag=1.5, tg=2.5, seed=42

In [15]:
TEXT_SWEEP_TRACK = 'mJS3'
TEXT_SWEEP_AUDIO = os.path.join(AUDIO_DIR, f'{TEXT_SWEEP_TRACK}.wav')
TEXT_SWEEP_PROMPTS = [
    'a person walks forward',
    'a person jumps',
    'a person kicks',
]
MBEAT_COLORS_TEXT = ['#F06292', '#64B5F6', '#81C784']

print('Generating 3 motions for text sweep...')
text_sweep_cells = []
for i, prompt in enumerate(TEXT_SWEEP_PROMPTS):
    print(f'  [{i+1}/3] "{prompt}"')
    motion = generate_motion(prompt, audio_path=TEXT_SWEEP_AUDIO, tg=2.5, ag=1.5, seed=42)
    joints = motion_to_joints(motion)
    cd = _prep_cell(joints, TEXT_SWEEP_AUDIO)
    text_sweep_cells.append(cd)
    print(f'        BAS = {cd["bas"]:.4f}')

print('\nRendering 1×3 grid video...')
text_sweep_labels = [f'"{p}"' for p in TEXT_SWEEP_PROMPTS]
genre_name, bpm = GENRES[TEXT_SWEEP_TRACK]
text_sweep_out = render_grid_video(
    text_sweep_cells, text_sweep_labels, MBEAT_COLORS_TEXT,
    title=f'Text Sweep — {genre_name} {bpm} BPM  |  ag=1.5  tg=2.5',
    out_path=os.path.join(OUTPUT_DIR, 'text_sweep_mJS3.mp4'),
    aud_path=TEXT_SWEEP_AUDIO,
)

display(Video(text_sweep_out, embed=True, width=900))

Generating 3 motions for text sweep...
  [1/3] "a person walks forward"


  0%|          | 0/1000 [00:00<?, ?it/s]

        BAS = 0.0575
  [2/3] "a person jumps"


  0%|          | 0/1000 [00:00<?, ?it/s]

        BAS = 0.1056
  [3/3] "a person kicks"


  0%|          | 0/1000 [00:00<?, ?it/s]

        BAS = 0.1505

Rendering 1×3 grid video...
  frame 1/160
  frame 41/160
  frame 81/160
  frame 121/160
  Saved → ./demo_outputs/text_sweep_mJS3.mp4


## §4 · Audio Sweep

Fixed prompt: **"a person dances"** · Col 0: text-only baseline (no audio) · Cols 1 & 2: pick genres from dropdown · ag=1.5, tg=2.5, seed=42  
*(Col 0 shows what the model generates from text alone — Cols 1 & 2 show how audio conditioning changes the motion.)*

In [16]:
audio_sweep_prompt = 'a person dances'

genre_keys = list(GENRE_OPTIONS.keys())

dd_col1 = widgets.Dropdown(options=genre_keys, value=genre_keys[0], description='Track 1:')
dd_col2 = widgets.Dropdown(options=genre_keys, value=genre_keys[2], description='Track 2:')
run_btn = widgets.Button(description='Generate & Render', button_style='success')
out_area = widgets.Output()

def _on_run(btn):
    with out_area:
        out_area.clear_output(wait=True)
        tk1 = GENRE_OPTIONS[dd_col1.value]
        tk2 = GENRE_OPTIONS[dd_col2.value]
        mbc = ['#9E9E9E', '#64B5F6', '#81C784']

        # Col 0: text-only baseline (no audio, ag=0)
        print('  [1/3] Generating text-only baseline (no audio)...')
        motion_base = generate_motion(audio_sweep_prompt, audio_path=None, tg=2.5, ag=0.0, seed=42)
        joints_base = motion_to_joints(motion_base)
        cd_base = _prep_cell(joints_base, aud_path=None)

        # Col 1: our model + track 1
        gn1, bpm1 = GENRES[tk1]
        aud1 = os.path.join(AUDIO_DIR, f'{tk1}.wav')
        print(f'  [2/3] Generating with {gn1} ({bpm1} BPM)...')
        motion1 = generate_motion(audio_sweep_prompt, audio_path=aud1, tg=2.5, ag=1.5, seed=42)
        joints1 = motion_to_joints(motion1)
        cd1 = _prep_cell(joints1, aud1)
        print(f'        BAS = {cd1["bas"]:.4f}')

        # Col 2: our model + track 2
        gn2, bpm2 = GENRES[tk2]
        aud2 = os.path.join(AUDIO_DIR, f'{tk2}.wav')
        print(f'  [3/3] Generating with {gn2} ({bpm2} BPM)...')
        motion2 = generate_motion(audio_sweep_prompt, audio_path=aud2, tg=2.5, ag=1.5, seed=42)
        joints2 = motion_to_joints(motion2)
        cd2 = _prep_cell(joints2, aud2)
        print(f'        BAS = {cd2["bas"]:.4f}')

        labels = ['Text-only (no audio)', f'{gn1} ({bpm1} BPM)', f'{gn2} ({bpm2} BPM)']
        cells = [cd_base, cd1, cd2]

        print('\nRendering 1×3 grid video (no audio - as we have multiple tracks)...')
        tag = f'base_{tk1}_{tk2}'
        out_path = render_grid_video(
            cells, labels, mbc,
            title=f'Audio Sweep — "{audio_sweep_prompt}"  |  ag=1.5  tg=2.5',
            out_path=os.path.join(OUTPUT_DIR, f'audio_sweep_{tag}.mp4'),
        )
        display(Video(out_path, embed=True, width=1000))

run_btn.on_click(_on_run)
display(widgets.HBox([dd_col1, dd_col2, run_btn]))
display(out_area)

Output()

## §5 · Baseline Comparisons

### 5a - Text-only: MDM vs Our Model
Same text prompt, our model also gets audio. MDM is the original pretrained MDM (no audio conditioning).

### 5b - Audio-only: EDGE vs Our Model
Same audio track, our model also gets a text prompt. EDGE outputs are pre-generated (8 AIST++ test songs).

In [ ]:
# ── §5a: Text-only — MDM baseline vs Our model ──────────────────────────────

_s5_models = {
    'Wav2CLIP + Beat-aware (best)': _resolve('stage2/audio_stage2_wav2clip_beataware/model_final.pt',
                                             './save/audio_stage2_wav2clip_beataware/model_final.pt'),
    'Wav2CLIP':                     _resolve('stage2/audio_stage2_wav2clip/model_final.pt',
                                             './save/audio_stage2_wav2clip/model_final.pt'),
    'Wav2CLIP + MOSPA':             _resolve('stage2/audio_stage2_wav2clip_mospa/model_final.pt',
                                             './save/audio_stage2_wav2clip_mospa/model_final.pt'),
    'Librosa-only (52-dim)':        _resolve('stage2/audio_stage2_librosa/model_final.pt',
                                             './save/audio_stage2_librosa/model_final.pt'),
}

_s5_loaded = {}

def _s5_get_model(path):
    if path not in _s5_loaded:
        print(f'  Loading {os.path.basename(os.path.dirname(path))} ...')
        m, m_args = load_model(path, DEVICE)
        _s5_loaded[path] = (m, m_args)
    return _s5_loaded[path]

genre_keys = list(GENRE_OPTIONS.keys())

# ── 5a widgets ───────────────────────────────────────────────────────────────
s5a_model  = widgets.Dropdown(options=list(_s5_models.keys()), value=list(_s5_models.keys())[0], description='Our model:')
s5a_genre  = widgets.Dropdown(options=genre_keys, value=genre_keys[0], description='Audio:')
s5a_prompt = widgets.Text(value='a person dances', description='Prompt:', layout=widgets.Layout(width='60%'))
s5a_btn    = widgets.Button(description='Run Text-only Comparison', button_style='success')
s5a_out    = widgets.Output()

def _on_s5a(btn):
    with s5a_out:
        s5a_out.clear_output(wait=True)
        prompt = s5a_prompt.value
        track = GENRE_OPTIONS[s5a_genre.value]
        gn, bpm = GENRES[track]
        aud = os.path.join(AUDIO_DIR, f'{track}.wav')
        model_path = _s5_models[s5a_model.value]

        # Col 0: Base MDM (text only, no audio)
        print(f'[1/2] MDM baseline: "{prompt}" (no audio)')
        motion_mdm = generate_motion_base_mdm(prompt, tg=2.5, seed=42)
        joints_mdm = motion_to_joints(motion_mdm)
        cd_mdm = _prep_cell(joints_mdm, aud_path=None)

        # Col 1: Our model (text + audio)
        m, m_args = _s5_get_model(model_path)
        feat_dim = m_args.get('audio_feat_dim', 519) if isinstance(m_args, dict) else getattr(m_args, 'audio_feat_dim', 519)
        print(f'[2/2] {s5a_model.value}: "{prompt}" + {gn} ({bpm} BPM)')

        torch.manual_seed(42); np.random.seed(42)
        cfg_m = AudioCFGSampleModel(m, text_scale=2.5, audio_scale=1.5)
        if feat_dim == 52:
            from model.audio_features_v2 import extract_audio_features_v2
            af = extract_audio_features_v2(aud, target_fps=FPS, duration=None)
            af = torch.from_numpy(af[:196]).float()
            beat_frames = list(np.where(af[:, 34] > 0.5)[0])
        else:
            af, beat_frames = extract_audio(aud, 196)
        n_frames = af.shape[0]
        audio_features = af.unsqueeze(0).to(DEVICE)
        mkw = {
            'y': {
                'text': [prompt],
                'mask': torch.ones(1, 1, 1, n_frames, dtype=torch.bool).to(DEVICE),
                'lengths': torch.tensor([n_frames]).to(DEVICE),
                'scale': torch.tensor([2.5]).to(DEVICE),
                'audio_features': audio_features,
                'beat_frames': beat_frames,
            }
        }
        with torch.no_grad():
            sample = diffusion.p_sample_loop(cfg_m, (1, 263, 1, n_frames),
                clip_denoised=False, model_kwargs=mkw, skip_timesteps=0,
                init_image=None, progress=True)
        motion_ours = sample.squeeze(2).permute(0, 2, 1).cpu().numpy()[0]
        motion_ours = motion_ours * STD + MEAN
        motion_ours = gaussian_filter1d(motion_ours, sigma=1.0, axis=0)
        joints_ours = motion_to_joints(motion_ours)
        cd_ours = _prep_cell(joints_ours, aud)
        print(f'  BAS (ours) = {cd_ours["bas"]:.4f}')

        labels = [f'MDM (text-only)', f'{s5a_model.value}']
        mbc = ['#9E9E9E', '#64B5F6']
        print('\nRendering 1×2 grid...')
        tag = f'textonly_{track}'
        out_path = render_grid_video(
            [cd_mdm, cd_ours], labels, mbc,
            title=f'Text-only: MDM vs Ours — "{prompt}"  |  audio={gn} {bpm} BPM',
            out_path=os.path.join(OUTPUT_DIR, f's5a_{tag}.mp4'),
        )
        display(Video(out_path, embed=True, width=800))

s5a_btn.on_click(_on_s5a)
display(widgets.HTML('<h4>5a · Text-only: MDM baseline vs Our model</h4>'))
display(widgets.VBox([s5a_model, s5a_prompt, s5a_genre, s5a_btn]))
display(s5a_out)

# ── §5b: Audio-only — EDGE baseline vs Our model ────────────────────────────

edge_songs = [f"{GENRES[k][0]} ({GENRES[k][1]} BPM)" for k in GENRES.keys()]
edge_song_map = {f"{v[0]} ({v[1]} BPM)": k for k, v in GENRES.items()}

s5b_model  = widgets.Dropdown(options=list(_s5_models.keys()), value=list(_s5_models.keys())[0], description='Our model:')
s5b_genre  = widgets.Dropdown(options=edge_songs, value=edge_songs[0], description='Audio:')
s5b_prompt = widgets.Text(value='a person dances', description='Prompt (ours):', layout=widgets.Layout(width='60%'))
s5b_btn    = widgets.Button(description='Run Audio-only Comparison', button_style='success')
s5b_out    = widgets.Output()

def _on_s5b(btn):
    with s5b_out:
        s5b_out.clear_output(wait=True)
        prompt = s5b_prompt.value
        track = edge_song_map[s5b_genre.value]
        gn, bpm = GENRES[track]
        aud = os.path.join(AUDIO_DIR, f'{track}.wav')
        model_path = _s5_models[s5b_model.value]

        # Col 0: EDGE (audio-only, pre-generated)
        print(f'[1/2] EDGE baseline: {gn} ({bpm} BPM) — loading pre-generated')
        edge_joints = load_edge_joints(track)
        cd_edge = _prep_cell_edge(edge_joints, aud, fps=EDGE_FPS)
        print(f'  BAS (EDGE) = {cd_edge["bas"]:.4f}  |  T={cd_edge["T"]} frames @ {EDGE_FPS} FPS')

        # Col 1: Our model (audio + text)
        m, m_args = _s5_get_model(model_path)
        feat_dim = m_args.get('audio_feat_dim', 519) if isinstance(m_args, dict) else getattr(m_args, 'audio_feat_dim', 519)
        print(f'[2/2] {s5b_model.value}: "{prompt}" + {gn} ({bpm} BPM)')

        torch.manual_seed(42); np.random.seed(42)
        cfg_m = AudioCFGSampleModel(m, text_scale=2.5, audio_scale=1.5)
        if feat_dim == 52:
            from model.audio_features_v2 import extract_audio_features_v2
            af = extract_audio_features_v2(aud, target_fps=FPS, duration=None)
            af = torch.from_numpy(af[:196]).float()
            beat_frames = list(np.where(af[:, 34] > 0.5)[0])
        else:
            af, beat_frames = extract_audio(aud, 196)
        n_frames = af.shape[0]
        audio_features = af.unsqueeze(0).to(DEVICE)
        mkw = {
            'y': {
                'text': [prompt],
                'mask': torch.ones(1, 1, 1, n_frames, dtype=torch.bool).to(DEVICE),
                'lengths': torch.tensor([n_frames]).to(DEVICE),
                'scale': torch.tensor([2.5]).to(DEVICE),
                'audio_features': audio_features,
                'beat_frames': beat_frames,
            }
        }
        with torch.no_grad():
            sample = diffusion.p_sample_loop(cfg_m, (1, 263, 1, n_frames),
                clip_denoised=False, model_kwargs=mkw, skip_timesteps=0,
                init_image=None, progress=True)
        motion_ours = sample.squeeze(2).permute(0, 2, 1).cpu().numpy()[0]
        motion_ours = motion_ours * STD + MEAN
        motion_ours = gaussian_filter1d(motion_ours, sigma=1.0, axis=0)
        joints_ours = motion_to_joints(motion_ours)
        cd_ours = _prep_cell(joints_ours, aud)
        print(f'  BAS (ours) = {cd_ours["bas"]:.4f}')

        labels = [f'EDGE (audio-only)', f'{s5b_model.value}']
        mbc = ['#FF8A65', '#64B5F6']
        print('\nRendering 1×2 grid...')
        tag = f'audioonly_{track}'
        out_path = render_grid_video(
            [cd_edge, cd_ours], labels, mbc,
            title=f'Audio-only: EDGE vs Ours — {gn} {bpm} BPM',
            out_path=os.path.join(OUTPUT_DIR, f's5b_{tag}.mp4'),
            aud_path=aud,
        )
        display(Video(out_path, embed=True, width=800))

s5b_btn.on_click(_on_s5b)
display(widgets.HTML('<h4>5b · Audio-only: EDGE baseline vs Our model</h4>'))
display(widgets.VBox([s5b_model, s5b_prompt, s5b_genre, s5b_btn]))
display(s5b_out)

HTML(value='<h4>5a · Text-only: MDM baseline vs Our model</h4>')

Output()

HTML(value='<h4>5b · Audio-only: EDGE baseline vs Our model</h4>')

Output()

## §6 · Free-Form Generation

Custom prompt · audio genre · model · text guidance · audio guidance · seed

In [18]:
STAGE2_MODELS = {
    'Wav2CLIP + Beat-aware (best)': _resolve('stage2/audio_stage2_wav2clip_beataware/model_final.pt',
                                             './save/audio_stage2_wav2clip_beataware/model_final.pt'),
    'Wav2CLIP':                     _resolve('stage2/audio_stage2_wav2clip/model_final.pt',
                                             './save/audio_stage2_wav2clip/model_final.pt'),
    'Wav2CLIP + MOSPA':             _resolve('stage2/audio_stage2_wav2clip_mospa/model_final.pt',
                                             './save/audio_stage2_wav2clip_mospa/model_final.pt'),
    'Librosa-only (52-dim)':        _resolve('stage2/audio_stage2_librosa/model_final.pt',
                                             './save/audio_stage2_librosa/model_final.pt'),
}

_loaded_models = {}

def _get_model(model_path):
    """Load and cache a model so switching is fast after the first load."""
    if model_path not in _loaded_models:
        print(f'Loading {model_path} ...')
        m, m_args = load_model(model_path, DEVICE)
        _loaded_models[model_path] = (m, m_args)
        print('  Done.')
    return _loaded_models[model_path]


def extract_audio_for_model(wav_path, n_frames, audio_feat_dim):
    """Extract audio features with the correct extractor for the model's expected dimension."""
    if audio_feat_dim == 52:
        from model.audio_features_v2 import extract_audio_features_v2
        feat = extract_audio_features_v2(wav_path, target_fps=FPS, duration=None)
        feat = feat[:n_frames]
        beat_idx = 34  # beat_soft channel in 52-dim layout
        beat_frames = list(np.where(feat[:, beat_idx] > 0.5)[0])
        return torch.from_numpy(feat).float(), beat_frames
    else:
        return extract_audio(wav_path, n_frames)


ff_model  = widgets.Dropdown(options=list(STAGE2_MODELS.keys()), value=list(STAGE2_MODELS.keys())[0], description='Model:')
ff_prompt = widgets.Text(value='a person dances energetically', description='Prompt:', layout=widgets.Layout(width='60%'))
ff_genre  = widgets.Dropdown(options=genre_keys, value=genre_keys[0], description='Audio:')
ff_tg     = widgets.FloatSlider(value=2.5, min=0.0, max=7.5, step=0.5, description='Text guidance:')
ff_ag     = widgets.FloatSlider(value=1.5, min=0.0, max=7.5, step=0.5, description='Audio guidance:')
ff_seed   = widgets.IntText(value=42, description='Seed:')
ff_btn    = widgets.Button(description='Generate & Visualize', button_style='primary')
ff_out    = widgets.Output()

def _on_ff(btn):
    with ff_out:
        ff_out.clear_output(wait=True)
        model_path = STAGE2_MODELS[ff_model.value]
        m, m_args = _get_model(model_path)
        feat_dim = m_args.get('audio_feat_dim', m_args.get('audio_feat_dim', 519)) if isinstance(m_args, dict) else getattr(m_args, 'audio_feat_dim', 519)

        track = GENRE_OPTIONS[ff_genre.value]
        gn, bpm = GENRES[track]
        aud = os.path.join(AUDIO_DIR, f'{track}.wav')
        prompt = ff_prompt.value
        tg, ag, seed = ff_tg.value, ff_ag.value, ff_seed.value

        torch.manual_seed(seed)
        np.random.seed(seed)

        cfg_model = AudioCFGSampleModel(m, text_scale=tg, audio_scale=ag)
        af, beat_frames = extract_audio_for_model(aud, 196, feat_dim)
        n_frames = af.shape[0]
        audio_features = af.unsqueeze(0).to(DEVICE)

        model_kwargs = {
            'y': {
                'text': [prompt],
                'mask': torch.ones(1, 1, 1, n_frames, dtype=torch.bool).to(DEVICE),
                'lengths': torch.tensor([n_frames]).to(DEVICE),
                'scale': torch.tensor([tg]).to(DEVICE),
                'audio_features': audio_features,
                'beat_frames': beat_frames,
            }
        }

        print(f'Generating: "{prompt}" + {gn} ({bpm} BPM)')
        print(f'Model: {ff_model.value} (feat_dim={feat_dim})  |  tg={tg}  ag={ag}  seed={seed}')
        with torch.no_grad():
            sample = diffusion.p_sample_loop(
                cfg_model, (1, 263, 1, n_frames),
                clip_denoised=False, model_kwargs=model_kwargs,
                skip_timesteps=0, init_image=None, progress=True,
                dump_steps=None, noise=None, const_noise=False,
            )
        motion = sample.squeeze(2).permute(0, 2, 1).cpu().numpy()[0]
        motion = motion * STD + MEAN
        motion = gaussian_filter1d(motion, sigma=1.0, axis=0)

        joints = motion_to_joints(motion)
        cd = _prep_cell(joints, aud)
        print(f'BAS = {cd["bas"]:.4f}')

        print('Rendering video...')
        short_name = ff_model.value.split('(')[0].strip().replace(' ', '_').lower()
        tag = f'freeform_{short_name}_s{seed}'
        out_path = render_grid_video(
            [cd], [f'"{prompt}"'], ['#F06292'],
            title=f'{ff_model.value}  |  {gn} {bpm} BPM  |  tg={tg}  ag={ag}',
            out_path=os.path.join(OUTPUT_DIR, f'{tag}.mp4'),
            aud_path=aud,
        )
        display(Video(out_path, embed=True, width=600))

ff_btn.on_click(_on_ff)
display(widgets.VBox([ff_model, ff_prompt, ff_genre, ff_tg, ff_ag, ff_seed, ff_btn]))
display(ff_out)

Output()